# Dive.ai — AI 인터랙티브 스토리 롤플레이 플랫폼

---

> **제출일**: 2026-05-15

## 목차

1. [프로젝트 개요](#1-프로젝트-개요)
2. [기술 스택](#2-기술-스택)
3. [사용 데이터](#3-사용-데이터)
   - 3-1. 문화콘텐츠 스토리 데이터 (AI Hub 145)
   - 3-2. 동아시아 고전 스토리 데이터 (AI Hub 70)
4. [데이터 전처리](#4-데이터-전처리)
5. [벡터 DB 구축 (RAG)](#5-벡터-db-구축-rag)
6. [시스템 아키텍처](#6-시스템-아키텍처)
7. [핵심 기능](#7-핵심-기능)
   - 7-11. 다시 하기
   - 7-26. 배포 기능
   - 7-27. 시나리오 갤러리
   - 7-28. 팔로잉 / 팔로워
   - 7-29. 작가 프로필
   - 7-30. 스토리 길이 선택 (단편 / 중편 / 장편)
8. [프론트엔드 화면 구성](#8-프론트엔드-화면-구성)
9. [결론](#9-결론)

---
## 1. 프로젝트 개요

**Dive.ai**는 AI 기반 인터랙티브 스토리 롤플레이 플랫폼입니다.

사용자가 장르·소재·캐릭터를 선택하면 AI가 **기승전결** 구조의 시나리오를 자동으로 설계하고,
그 서사 위에서 AI 캐릭터와 1:1 채팅 롤플레이를 진행할 수 있습니다.

### 핵심 목표

| 과제 | 설명 |
|---|---|
| **콘텐츠 생성 파이프라인** | 유저 입력을 받아 RAG 기반 씬 패턴 검색과 LLM 생성을 조합하여 시나리오·캐릭터·나침반·로어북을 자동 생성 |
| **서사 일관성 유지 채팅** | 생성된 나침반(Compass)을 기반으로 대화 단계(기승전결)를 추적하면서 이탈 감지·단계 전환·엔딩 생성을 자동 처리 |

### 주요 특징

- 실시간 SSE 스트리밍으로 AI 응답을 문장 단위로 즉시 표시
- 89,851개 씬 패턴 + 12,717개 고전 단락 데이터 기반 RAG
- Google Firebase 인증 + 토큰 시스템 + 30일 연속 출석체크
- 이미지(캐릭터/배경/엔딩) + BGM + 시네마틱 영상 생성

---
## 2. 기술 스택

### 백엔드

| 기술 | 버전 | 역할 |
|---|---|---|
| **Python** | 3.11 | LangChain·ChromaDB 생태계 호환 |
| **FastAPI** | - | API 라우팅, 비동기 SSE 스트리밍 |
| **SQLAlchemy + SQLite** | - | ORM 기반 스키마 관리 |
| **LangChain** | - | Vertex AI·OpenAI 통합 인터페이스 |
| **ChromaDB** | - | 임베딩 벡터 로컬 저장·검색 |
| **Firebase Admin SDK** | - | Google 로그인 토큰 검증 |

### AI 모델

| 모델 | 모델 ID | 용도 |
|---|---|---|
| **Gemini 3 Flash** | `gemini-3-flash-preview` | 채팅 응답, 빌더 파이프라인 기본 모델 |
| **Gemini 2.5 Pro** | `gemini-2.5-pro-vertex` | 채팅 고품질 모델 (유저 선택) |
| **Gemini 3.1 Pro** | `gemini-3.1-pro-preview` | 엔딩 씬·시나리오·나침반 재생성 등 고품질 기능 |
| **Gemini 3.1 Flash-Lite** | `gemini-3.1-flash-lite-preview` | 힌트 카드·요약 등 경량 반복 처리 |
| **Gemini 3.1 Flash Image** | `gemini-3.1-flash-image-preview` | 이미지 생성 (캐릭터/배경/엔딩/표지) |
| **Veo 3.1 Fast** | `veo-3.1-fast-generate-001` | 시네마틱 영상 생성 |
| **Lyria 3 Pro** | `lyria-3-pro-preview` | BGM 생성 |
| **text-embedding-3-small** (OpenAI) | - | RAG 씬 검색용 임베딩 (1536차원) |

### 프론트엔드

| 기술 | 역할 |
|---|---|
| **React 18 + TypeScript** | 컴포넌트 재사용성, 타입 안정성 |
| **Vite** | 빠른 개발 서버 및 빌드 |
| **Tailwind CSS** | 별도 CSS 파일 없이 빠른 UI 프로토타이핑 |

---
## 3. 사용 데이터

Dive.ai는 두 가지 AI Hub 공개 데이터셋을 활용합니다.

| 데이터셋 | 출처 | 규모 | 역할 |
|---|---|---|---|
| **문화콘텐츠 스토리 데이터** | AI Hub 145 | 3,561개 작품 / **89,851개 씬** | 모든 장르의 서사 구조 학습 (주 데이터) |
| **동아시아 고전 스토리 데이터** | AI Hub 70 | 816개 작품 / **12,717개 단락** | 고전 장르 선택 시 세계관·분위기 보강 (보조 데이터) |

### 3-1. 문화콘텐츠 스토리 데이터 (AI Hub 145)

**구성**: 영화·시리즈·소설·만화 4개 카테고리 / Training + Validation / 원천데이터(JSON) + 라벨링데이터(JSON)

```
145.다양한_문화콘텐츠_스토리_데이터/
└── 01-1.정식개방데이터/
    ├── Training/
    │   ├── 01.원천데이터/  (TS_01.영화 / TS_02.시리즈 / TS_03.소설 / TS_04.만화)
    │   └── 02.라벨링데이터/ (TL_01 ~ TL_04)
    └── Validation/
        ├── 01.원천데이터/  (VS_01 ~ VS_04)
        └── 02.라벨링데이터/ (VL_01 ~ VL_04)
```

**라벨링데이터 주요 필드 (작품 레벨)**

| 필드 | 설명 |
|---|---|
| `genre` | 장르 배열 (드라마, 멜로/로맨스, 스릴러, 판타지 등 10종) |
| `theme` | 작품 핵심 테마 (사랑, 도전, 복수 등 22종) |
| `motif` | 작품 모티프 (삼각관계, 짝사랑 등 179종) |
| `units` | 씬(unit) 단위 분석 배열 |

**라벨링데이터 주요 필드 (씬 레벨)**

| 필드 | 설명 |
|---|---|
| `stage` | 서사 프레임워크 단계 레이블 (스토리헬퍼 16단계 또는 영웅의 여정 12단계) |
| `unit_motif` | 씬 단위 모티프 (부탁/제안, 걱정 등 403종) |
| `storyline` | 씬 내용 요약 텍스트 → **임베딩 핵심 필드** |

In [ ]:
import json
import os
from pathlib import Path
from collections import Counter

# 전처리 완료된 씬 인덱스 로드
OUTPUT_DIR = Path("output")
with open(OUTPUT_DIR / "scene_metadata_index.json", encoding="utf-8") as f:
    scene_index = json.load(f)

print(f"총 씬 수: {len(scene_index):,}개")
print(f"\n첫 번째 씬 샘플:")
sample = scene_index[0]
for k, v in sample.items():
    print(f"  {k}: {v}")

In [ ]:
# 카테고리별 씬 수
cat_counter = Counter(s["category_name"] for s in scene_index)
print("카테고리별 씬 수:")
for cat, cnt in cat_counter.most_common():
    print(f"  {cat}: {cnt:,}개")

print()

# 프레임워크별 씬 수
fw_counter = Counter(s["framework"] for s in scene_index)
print("프레임워크별 씬 수:")
for fw, cnt in fw_counter.most_common():
    print(f"  {fw}: {cnt:,}개")

In [ ]:
# 장르별 통계 (genre_stats.json)
with open(OUTPUT_DIR / "genre_stats.json", encoding="utf-8") as f:
    genre_stats = json.load(f)

print("장르별 작품 수 및 씬 수:")
print(f"  {'장르':<15} {'작품 수':>8} {'씬 수':>10}")
print("  " + "-" * 35)
for genre, stats in sorted(genre_stats.items(), key=lambda x: x[1]["work_count"], reverse=True):
    print(f"  {genre:<15} {stats['work_count']:>8,} {stats['scene_count']:>10,}")

#### 서사 프레임워크 — 두 가지 구조 체계

데이터셋은 두 가지 서사 프레임워크를 사용하며, 씬마다 `framework` 필드로 구분됩니다.

| 프레임워크 | 씬 수 | 비율 |
|---|---:|---:|
| **스토리헬퍼 16단계** (한국콘텐츠진흥원 KOCCA) | 88,424개 | 98.4% |
| **영웅의 여정 12단계** (조지프 캠벨 Hero's Journey) | 1,427개 | 1.6% |

---

#### Dive.ai에서의 활용 방식

두 프레임워크는 **사용되는 시점이 다릅니다**.

| 시점 | 사용 프레임워크 | 이유 |
|---|---|---|
| **시나리오 생성 (빌더 파이프라인)** | 스토리헬퍼 씬만 사용 | `story_arc="기"` 필터 적용 → "Opening Salvo / Main Character / Setting-up" 단계 씬만 검색 |
| **채팅 진행 중 씬 컨텍스트** | 스토리헬퍼 + 영웅의 여정 모두 | stage 필터 없이 의미 유사도로만 검색하므로 두 프레임워크 씬 모두 후보에 포함 |

**시나리오 생성 단계**에서는 `STORY_ARC_STAGES` 딕셔너리가 기승전결을 스토리헬퍼 단계명으로 매핑하여, 현재 서사 단계에 맞는 씬만 좁혀서 검색합니다.

```python
# scenario_builder.py
STORY_ARC_STAGES = {
    "기": ["Opening Salvo", "Main Character", "Setting-up"],
    "승": ["1st Accident", "Villains Move", "Doubts & Debate", "Making a Choice", "Choice to Fight"],
    "전": ["Ups & Downs", "2nd Accident", "Innermost Cave", "Defeat", "Resurrection", "Another Story"],
    "결": ["Trailer Moments", "Final Salvo"],
}
# story_arc="기" 로 호출 시 → ChromaDB 필터: {"stage": {"$in": ["Opening Salvo", "Main Character", "Setting-up"]}}
# hero_journey 씬("일상세계" 등)은 이 필터에 매칭되지 않아 시나리오 생성 단계에서는 제외됨
```

**채팅 진행 중**에는 stage 필터 없이 소재+대화 내용의 의미 유사도만으로 검색하므로, 영웅의 여정 씬도 소재가 유사하면 채팅 컨텍스트로 활용될 수 있습니다.

---

#### 스토리헬퍼 16단계

한국콘텐츠진흥원(KOCCA)의 스토리 창작 도구. 영화·드라마 서사를 16개 단계로 분석합니다.

| 순서 | stage | 의미 | 씬 수 |
|---|---|---|---:|
| 1 | `Opening Salvo` | 첫 장면, 세계관 제시 | 3,745 |
| 2 | `Main Character` | 주인공 소개 | 4,865 |
| 3 | `Setting-up` | 배경·관계 설정 | 8,380 |
| 4 | `1st Accident` | 첫 번째 사건 | 7,471 |
| 5 | `Villains Move` | 적대자 본격 등장 | 6,516 |
| 6 | `Doubts & Debate` | 주인공 갈등·고민 | 7,139 |
| 7 | `Making a Choice` | 주인공 결단 | 7,394 |
| 8 | `Choice to Fight` | 행동 시작 | 6,292 |
| 9 | `Ups & Downs` | 갈등과 소소한 성공/실패 반복 | 4,105 |
| 10 | `2nd Accident` | 두 번째 위기 | 7,658 |
| 11 | `Innermost Cave` | 가장 위험한 상황 | 4,546 |
| 12 | `Defeat` | 주인공 패배·최대 위기 | 5,446 |
| 13 | `Resurrection` | 패배를 딛고 재기 | 3,963 |
| 14 | `Another Story` | 서브플롯 전개 | 3,836 |
| 15 | `Trailer Moments` | 클라이맥스 직전 긴장 최고조 | 3,492 |
| 16 | `Final Salvo` | 모든 갈등 해소·마무리 | 3,576 |

---

#### 영웅의 여정 12단계 (Hero's Journey)

조지프 캠벨의 신화 구조 이론 기반. 판타지·액션 장르에 주로 분포합니다.

| 순서 | stage | 씬 수 |
|---|---|---:|
| 1 | `일상세계` | 144 |
| 2 | `모험에의 소명` | 207 |
| 3 | `소명부활의 거부` | 44 |
| 4 | `정신적 스승과의 만남` | 82 |
| 5 | `첫 관문의 통과` | 72 |
| 6 | `시험, 협력자, 적대자` | 299 |
| 7 | `동굴 가장 깊은 곳으로의 진입` | 116 |
| 8 | `시련` | 182 |
| 9 | `보상` | 36 |
| 10 | `귀환의 길` | 61 |
| 11 | `부활` | 89 |
| 12 | `영약을 가지고 귀환` | 95 |

### 3-2. 동아시아 고전 스토리 데이터 (AI Hub 70)

**구성**: 한국(71개 작품) / 중국(319개) / 일본(426개) / 총 816개 작품 / 12,717개 단락

**원천데이터**: `.txt` (단락 단위 한국어 번역문)

**라벨링데이터** `_info.json` 주요 필드:

| 필드 | 설명 |
|---|---|
| `내용분류.장르` | 설화, 가문소설, 무협, 판타지, 호러 등 |
| `내용분류.사건/모티프` | 기적, 복수, 귀신, 전투 등 |
| `내용분류.공간` | 궁, 전장, 산, 바다 등 |
| `내용분류.캐릭터` | 왕족/귀족, 승려, 요괴 등 |
| `주제문` | 단락 내용 1문장 요약 → **임베딩 핵심 필드** |

**Dive.ai에서의 활용**: 유저가 "조선", "중국 무협", "일본 설화" 등 고전 장르를 선택했을 때만 RAG 검색에 포함됩니다.

In [ ]:
# 동아시아 고전 단락 인덱스 로드
with open(OUTPUT_DIR / "classic_paragraph_index.json", encoding="utf-8") as f:
    classic_index = json.load(f)

print(f"총 단락 수: {len(classic_index):,}개")

# 국가별 단락 수
country_counter = Counter(p["country"] for p in classic_index)
print("\n국가별 단락 수:")
for country, cnt in country_counter.most_common():
    print(f"  {country}: {cnt:,}개")

# 샘플
print("\n샘플 단락 (한국):")
sample_kor = next(p for p in classic_index if p["country"] == "한국")
print(f"  장르: {sample_kor['genre']}")
print(f"  모티프: {sample_kor['motif']}")
print(f"  공간: {sample_kor['space']}")
print(f"  주제문: {sample_kor['summary'][:80]}...")

---
## 4. 데이터 전처리

원본 AI Hub 데이터를 ChromaDB 임베딩에 적합한 형태로 변환합니다.

### 전처리 출력 파일

| 파일 | 내용 |
|---|---|
| `output/scene_metadata_index.json` | 문화콘텐츠 씬 단위 플랫 인덱스 (89,851개) |
| `output/work_metadata_index.json` | 작품 단위 메타데이터 인덱스 |
| `output/classic_paragraph_index.json` | 동아시아 고전 단락 인덱스 (12,717개) |
| `output/genre_stats.json` | 장르별 통계 |
| `output/classic_country_stats.json` | 국가별 고전 통계 |

### 임베딩 텍스트 구성 전략

임베딩에 사용할 텍스트는 **의미적 유사도 검색**에 최적화된 필드를 선택합니다.

| 데이터 | 임베딩 텍스트 | 이유 |
|---|---|---|
| 문화콘텐츠 씬 | `unit_motif + ": " + storyline` | 씬의 핵심 서사 의미를 가장 잘 표현 |
| 동아시아 고전 단락 | `주제문` | 단락 내용 1문장 요약 |

> `causality` (다음 씬과의 인과 관계)는 씬 자체 내용이 아니므로 임베딩에서 제외합니다.

In [ ]:
# 임베딩 텍스트 구성 함수 (실제 구현과 동일)
def build_scene_text(scene: dict) -> str:
    """씬 임베딩 텍스트 구성: unit_motif + storyline"""
    motif = scene.get("unit_motif") or ""
    storyline = scene.get("storyline") or ""
    if motif and storyline:
        return f"{motif}: {storyline}"
    return storyline or motif

def build_classic_text(paragraph: dict) -> str:
    """고전 단락 임베딩 텍스트: 주제문"""
    return paragraph.get("summary") or ""

# 실제 임베딩 텍스트 샘플 출력
print("[씬 임베딩 텍스트 샘플]")
for scene in scene_index[:3]:
    text = build_scene_text(scene)
    print(f"  • [{scene['stage']}] {text[:80]}")

print()
print("[고전 단락 임베딩 텍스트 샘플]")
for para in classic_index[:3]:
    text = build_classic_text(para)
    print(f"  • [{para['country']}] {text[:80]}")

In [ ]:
# 스토리헬퍼 16단계 씬 수 분포 시각화
storyhelper_scenes = [s for s in scene_index if s["framework"] == "storyhelper"]
stage_counter = Counter(s["stage"] for s in storyhelper_scenes)

# 서사 순서로 정렬
stage_order = [
    "Opening Salvo", "Main Character", "Setting-up", "1st Accident",
    "Villains Move", "Doubts & Debate", "Making a Choice", "Choice to Fight",
    "Ups & Downs", "2nd Accident", "Innermost Cave", "Defeat",
    "Resurrection", "Another Story", "Trailer Moments", "Final Salvo"
]

print("스토리헬퍼 16단계별 씬 수:")
for stage in stage_order:
    cnt = stage_counter.get(stage, 0)
    bar = "█" * (cnt // 200)
    print(f"  {stage:<20} {cnt:>6,}  {bar}")

---
## 5. 벡터 DB 구축 (RAG)

### 구조

| 항목 | 내용 |
|---|---|
| **임베딩 모델** | `text-embedding-3-small` (OpenAI, 1536차원) |
| **벡터 DB** | ChromaDB PersistentClient (`vectordb/` 폴더에 영구 저장) |
| **컬렉션 1** | `scenes` — 문화콘텐츠 씬 패턴 (89,851개) |
| **컬렉션 2** | `classics` — 동아시아 고전 단락 (12,717개) |
| **예상 비용** | `text-embedding-3-small` 기준 약 $0.07 (1회 구축) |

### RAG 파이프라인

```
유저 입력 (소재 + 장르)
        ↓
[Step 1] OpenAI text-embedding-3-small (1536차원)으로 소재 쿼리 벡터 생성
        ↓
[Step 2] ChromaDB scenes 컬렉션에서 코사인 유사도 기준
         상위 K×15개(기본 75개) 후보 씬 검색
         → 이 단계에서는 장르 무관, 의미적으로 유사한 씬을 넓게 수집
        ↓
[Step 3] Python 레벨 장르 post-filter
         → 75개 후보 중 선택한 장르가 포함된 씬만 남김
         → 단, 장르 미선택 시 이 단계 생략
        ↓
[Step 4] 최종 상위 K개(기본 5개) 씬 반환 → LLM 프롬프트에 주입
        ↓
[Step 5] Gemini 3 Flash (gemini-3-flash-preview)로 시나리오 생성
```

> **핵심**: 시나리오는 특정 장르에서 *자주 등장하는 패턴*이 아니라, **소재와 의미적으로 가장 유사한 씬**을 먼저 넓게 검색한 뒤 장르 조건으로 추려낸 결과를 참고해 생성됩니다.

#### 고전(동아시아) 콘텐츠 유형의 RAG

고전 콘텐츠 유형을 선택한 경우에도 씬 패턴 검색은 동일하게 `scenes` 컬렉션 전체에서 진행합니다.  
추가로 **국가(나라)를 선택했을 때만** `classics` 컬렉션에서 해당 국가의 고전 단락을 검색하여 세계관·분위기 보강 컨텍스트로 함께 주입합니다.

```python
# scenario_builder.py (핵심 로직 요약)
def retrieve_scenes(query, genre=None, top_k=5):
    genre_db = _normalize_genre(genre) if genre else None
    query_vec = openai_client.embeddings.create(
        model="text-embedding-3-small", input=[query]
    ).data[0].embedding

    # Step 2: 의미 유사도로 K×15개 후보 수집 (장르 필터 없음)
    fetch_k = top_k * 15 if genre_db else top_k
    results = scene_collection.query(query_embeddings=[query_vec], n_results=fetch_k)

    scenes = []
    for meta, doc, dist in zip(results["metadatas"][0],
                               results["documents"][0],
                               results["distances"][0]):
        # Step 3: Python 레벨 장르 post-filter
        if genre_db and genre_db not in meta.get("genre", ""):
            continue
        scenes.append({"content": doc, "metadata": meta, "score": 1 - dist})
        if len(scenes) >= top_k:
            break
    return scenes  # Step 4: 최종 K개 반환

# 고전 컨텍스트는 country 선택 시에만 추가 (세계관·분위기 보강)
if country and country in CLASSIC_COUNTRIES:
    classics = retrieve_classics(query, country=country, top_k=3)
    context += "\n\n" + build_classic_context(classics)
```

In [ ]:
import chromadb
from pathlib import Path

# ChromaDB 연결 (실제 구현과 동일 경로)
DB_DIR = Path("vectordb")
chroma_client = chromadb.PersistentClient(path=str(DB_DIR))

scene_collection = chroma_client.get_collection("scenes")
classic_collection = chroma_client.get_collection("classics")

print(f"scenes  컬렉션: {scene_collection.count():,}개")
print(f"classics 컬렉션: {classic_collection.count():,}개")

In [ ]:
# RAG 검색 함수 (scenario_builder.py 실제 구현)
# ※ OpenAI API 키 필요. 없으면 RAG 없이 Gemini 단독으로 시나리오 생성 (Fallback 구현됨)

def retrieve_scenes_demo(query: str, genre: str = None, top_k: int = 5):
    """
    씬 RAG 검색 예시 (실제 구현과 동일 로직)
    
    - 쿼리를 text-embedding-3-small로 임베딩
    - ChromaDB에서 코사인 유사도 기반 top_k * 15개 후보 검색
    - 장르 필터링 후 최종 top_k개 반환
    """
    try:
        from openai import OpenAI
        client = OpenAI()
        EMBED_MODEL = "text-embedding-3-small"
        
        query_vec = client.embeddings.create(model=EMBED_MODEL, input=[query]).data[0].embedding
        
        genre_map = {"드라마": "드라마", "멜로": "멜로/로맨스", "판타지": "판타지"}
        genre_db = genre_map.get(genre, genre)
        
        fetch_k = min(top_k * 15 if genre_db else top_k, scene_collection.count())
        results = scene_collection.query(
            query_embeddings=[query_vec],
            n_results=fetch_k,
            include=["metadatas", "documents", "distances"]
        )
        
        scenes = []
        for meta, doc, dist in zip(
            results["metadatas"][0],
            results["documents"][0],
            results["distances"][0]
        ):
            if genre_db and genre_db not in meta.get("genre", ""):
                continue
            scenes.append({
                "score": round(1 - dist, 4),
                "stage": meta.get("stage"),
                "unit_motif": meta.get("unit_motif"),
                "storyline": doc
            })
            if len(scenes) >= top_k:
                break
        return scenes
    except ImportError:
        print("openai 패키지 없음 — pip install openai")
        return []

print("retrieve_scenes_demo 함수 정의 완료")
print("실행 예시: retrieve_scenes_demo('복수극 대결 장면', genre='스릴러', top_k=3)")

---
## 6. 시스템 아키텍처

### 전체 구조

```
┌─────────────────────────────────────────────────────────────┐
│                    Frontend (React 18 + TypeScript)          │
│  Screen0~5 (빌더 플로우)  │  ChatInterface  │  ChatList     │
│  Settings  │  AttendanceScreen  │  LoginScreen  │ NicknameScreen │
└──────────────┬──────────────────┬──────────────────────────┘
               │ POST /builder/run│ POST /chat/stream
               │      (SSE)       │      (SSE)
┌──────────────▼──────────────────▼──────────────────────────┐
│               main.py (FastAPI)                              │
│  라우팅·CORS·Firebase 인증·SQLite DB·SSE 스트리밍            │
└───┬───────────────┬──────────────────────┬──────────────────┘
    │               │                      │
    ▼               ▼                      ▼
scenario_       chat_engine_v2        prompt_builder
builder.py      .py                   .py + memory.py
(빌더 파이프라인)  (v2 채팅 엔진)       (레거시 채팅)
    │               │
    └───────┬───────┘
            ▼
        ai_engine.py
        (LLM 팩토리)
            │
     ┌──────┴──────┐
     ▼             ▼
 Vertex AI      OpenAI
 (Gemini 등)    (임베딩)
                   │
               ChromaDB (RAG)
```

### 주요 API 엔드포인트

| 엔드포인트 | 메서드 | 설명 |
|---|---|---|
| `/builder/run` | POST (SSE) | 시나리오 생성 파이프라인 (6단계) |
| `/chat/stream` | POST (SSE) | 채팅 스트리밍 (v2 / 레거시 자동 분기) |
| `/auth/google` | POST | Firebase Google 로그인 → 내부 JWT 발급 |
| `/auth/checkin` | POST | 일일 출석체크 (+5 DT, 하루 1회) |
| `/auth/attendance` | GET | 30일 연속 출석 현황 조회 |
| `/topics/{id}/duplicate` | POST | 대화 분기 (Fork) |
| `/topics/{id}/relationship-graph/refresh` | POST | 관계도 수동 갱신 |
| `/topics/{id}/inner-thought` | POST | 속마음 생성·갱신 |
| `/topics/{id}/suggest-replies` | GET | AI 추천 답변 3개 |
| `/topics/{id}/generate-background` | POST | 배경 이미지 생성 |
| `/topics/{id}/cinematic` | POST | 시네마틱 영상 생성 |
| `/chat/generate-bgm` | POST | BGM 생성 |

### 채팅 엔진 분기 로직

```python
# main.py — POST /chat/stream
if topic.compass:          # 빌더 파이프라인을 통해 생성된 토픽
    await _chat_stream_v2()   # GameStateV2 + 나침반 기반
else:                      # 수동 생성 토픽
    generate_legacy()         # LangChain 스트리밍 (하위 호환)
```

---
## 7. 핵심 기능

---
### 7-1. 빌더 파이프라인

유저가 콘텐츠 유형·장르·소재·캐릭터를 입력하면 SSE로 6단계 진행 상황을 실시간 전송하며 시나리오를 생성합니다.

```
Step 1: 소재 자동 분석    → generate_query_auto()         [직전 소재 중복 방지]
Step 2: 시나리오 작성     → generate_scenario_with_rag()  [RAG + Gemini]
Step 3: 캐릭터 설계      → generate_characters()          [이름 중복 방지]
Step 4: 나침반 생성      → generate_compass()              [캐릭터 정보 반영]
Step 5: 로어북 + 관계도   → extract_lorebook_entries()
                           generate_relationship_graph()
Step 6: 시작 배경 요약    → generate_intro_display()       [3~4문장, 캐릭터 이름 주입]
```

**구현 위치**: `main.py` → `builder_run()` / `scenario_builder.py`

**SSE 이벤트 형식** (`_sse()` 헬퍼로 통일):
```python
# _sse(dict) → "data: {json}\n\n"
yield _sse({"step": 1, "label": "소재 분석 중..."})
yield _sse({"step": 6, "done": True, "topic_id": topic.id, "intro_display": "..."})
```

**OpenAI 키 없을 때 Fallback**: `OPENAI_API_KEY`가 없으면 RAG 컨텍스트를 빈 문자열로 처리하고 Vertex AI 단독으로 시나리오를 생성합니다.

---
### 7-1-1. 콘텐츠 유형 / 장르 선택

빌더 Step 1(Screen1.tsx)에서 유저가 **콘텐츠 유형**과 **장르**를 선택합니다.  
이 두 값은 이후 시나리오 생성·캐릭터 디자인·이미지 스타일·RAG 검색 등 전체 파이프라인에 일관되게 반영됩니다.

**구현 위치**: `frontend/src/components/Screen1.tsx`, `scenario_builder.py`

---

#### 콘텐츠 유형 (5종)

| 콘텐츠 유형 | 이미지 스타일 | 비고 |
|---|---|---|
| **만화** | 한국 웹툰 manhwa 일러스트 (2D 그림체, 선명한 셀 쉐이딩) | |
| **소설** | 웹소설 일러스트 스타일 (부드러운 조명, 감성적 색감) | |
| **시리즈** | 실사 HD 사진 (드라마 스튜디오 촬영 느낌) | |
| **영화** | 시네마틱 필름 스틸 (영화 조명, 필름 그레인) | |
| **고전** | 역사 웹툰 스타일 (국가별 시대 배경 반영) | 국가 선택 추가 필요 |

- **무작위(Shuffle)** 버튼: 고전을 제외한 4종 중 무작위로 유형과 장르를 동시에 선택

---

#### 장르 선택

콘텐츠 유형별로 선택 가능한 장르 목록이 다릅니다.

**만화 / 시리즈 / 영화** (각 10종):

> 드라마, 멜로·로맨스, 스릴러, 판타지, 액션, 미스터리, 코미디, SF, 전쟁, 공포(호러)

**소설** (9종, 공포 제외):

> 드라마, 멜로·로맨스, 스릴러, 판타지, 액션, 미스터리, 코미디, SF, 전쟁

**고전** — 국가 선택 후 해당 국가 장르 표시:

| 국가 | 선택 가능 장르 |
|---|---|
| **한국** | 가문소설, 판타지, 로맨스, 영웅소설, 미스터리, 공포(호러) |
| **중국** | 무협, 로맨스, 공포(호러), 판타지, 미스터리 |
| **일본** | 설화, 미스터리, 공포(호러) |

- 고전 유형은 국가를 선택해야 장르 목록이 표시되고, **국가+장르 모두 선택**해야 다음 단계로 진행 가능

---

#### 콘텐츠 유형이 파이프라인에 미치는 영향

| 영역 | 만화 / 소설 | 시리즈 / 영화 | 고전 |
|---|---|---|---|
| **캐릭터 이미지** | `_SYSTEM_WEBTOON` (2D 웹툰) | 실사 사진 | `_SYSTEM_WEBTOON_CLASSIC` (역사 웹툰) |
| **표지 이미지** | 웹툰 manhwa 구도 | 드라마/영화 포스터 구도 | 역사 웹툰 + 시대 배경 |
| **고전 시대 설정** | — | — | 한국→조선(Joseon) / 중국→명청(Ming/Qing) / 일본→에도(Edo) |
| **캐릭터 이름 생성** | `{genre} 세계관에 어울리는 이름` | 동일 | `{country} {genre} 배경의 실제 이름` |
| **RAG 검색** | scenes + 장르 필터 | scenes + 장르 필터 | scenes + classics(국가 필터) 병행 |
| **시나리오 프롬프트** | `{category}용 {genre} 장르에 최적화된 기승전결` | 동일 | 동일 + 고전 세계관 컨텍스트 |

#### 장르 정규화 처리

프론트엔드 표기와 ChromaDB 메타데이터 키가 일치하지 않는 일부 장르는 백엔드에서 자동 변환됩니다.

```python
# scenario_builder.py
_GENRE_NORMALIZE = {
    "멜로·로맨스": "멜로/로맨스",   # UI 표기 → DB 키
    "공포(호러)":  "호러",           # UI 표기 → DB 키
}
```

---
### 7-2. v2 채팅 엔진 — GameStateV2

채팅 엔진의 핵심은 **기승전결 4단계 상태머신**입니다.
매 턴마다 LLM 응답에서 JSON을 파싱하여 상태를 자동 갱신합니다.

**구현 위치**: `chat_engine_v2.py`

#### GameStateV2 구조

In [ ]:
import sys
sys.path.insert(0, "Dive.ai_prototype")

from chat_engine_v2 import GameStateV2, MIN_STAGE_TURNS, OFF_TRACK_THRESHOLD

print("=== GameStateV2 상태 구조 ===")
gs = GameStateV2()
print(f"초기 단계: {gs.current_stage}")
print(f"초기 호감도: {gs.affinity}")
print(f"총 턴 수: {gs.total_turn_count}")
print()
print("=== 단계별 최소 턴 수 ===")
for stage, turns in MIN_STAGE_TURNS.items():
    print(f"  {stage} 단계: 최소 {turns}턴")
print()
print(f"이탈 감지 임계값: {OFF_TRACK_THRESHOLD}회 연속 이탈 시 나침반 재생성")
print()
print("=== 호감도 시스템 ===")
print("범위: -100 ~ +100")
print("기 단계 delta cap: ±3")
print("승 단계 delta cap: ±5")
print("전 단계 delta cap: ±8")
print()
print("=== 상태 직렬화 ===")
print(gs.to_dict())

#### 채팅 1턴 처리 흐름

```
유저 메시지 수신
  ↓
1. get_lorebook_context()    [키워드 매칭 + foreshadowing 항상 포함]
  ↓
2. build_chat_system_prompt() [나침반 + GameStateV2 + 로어북 + 관계도 + 요약 주입]
  ↓
3. Vertex AI / GPT 호출 (스트리밍)
  ↓
4. SSE chunk 전송 (문장 단위)
  ↓
5. parse_chat_response()     [JSON 파싱: affinity_delta, trigger_branch, trigger_ending, off_track]
  ↓
6. Stage Gate 체크            [MIN_STAGE_TURNS 미충족 시 trigger_branch/ending 강제 False]
  ↓
7-a. trigger_branch == True → generate_next_stage()  [단계 전환 오프닝]
7-b. off_track >= 3 연속    → regenerate_compass()   [나침반 재생성]
7-c. trigger_ending == True → generate_ending_scene() [호감도 기반 엔딩]
  ↓
8. 10턴마다 check_and_auto_summarize()  [요약 자동 생성·저장]
  ↓
9. 단계 전환 시 generate_relationship_graph() [관계도 갱신]
  ↓
10. DB 저장 (game_state, message, relationship_graph)
  ↓
11. SSE done 이벤트 전송 {affinity, stage, hint_card, inner_thoughts, ...}
```

#### 엔딩 유형 (호감도 기반)

| 호감도 | 엔딩 유형 |
|---|---|
| +70 ~ +100 | 해피 엔딩 |
| +35 ~ +69 | 중립 엔딩 |
| -100 ~ +34 | 배드 엔딩 |

엔딩 이후에는 `GameStateV2.is_ended = True`로 설정되어 추가 채팅이 차단됩니다.

---
### 7-3. 나침반 (Compass)

나침반은 AI가 서사의 방향성을 잃지 않도록 하는 **서사 가이드**입니다.  
빌더 파이프라인에서 생성되며 채팅 시스템 프롬프트에 매 턴 주입됩니다.

**구현 위치**: `chat_engine_v2.py` → `generate_compass()`

**나침반 JSON 구조**:
```json
{
  "narrative_goal": "서사 전체 목표",
  "conflict_core": "핵심 갈등 구조",
  "foreshadowing": ["복선 1", "복선 2"],
  "trigger_conditions": ["단계 전환 조건들"],
  "stage_constraints": {
    "기": "기 단계에서 반드시 일어나야 할 것",
    "승": "승 단계 지침",
    "전": "전 단계 지침",
    "결": "결말 방향성"
  }
}
```

**이탈 감지**: `off_track` 플래그가 3턴 연속으로 True이면 `regenerate_compass()`를 호출하여 나침반을 자동 재생성합니다.

**복선 처리**: `foreshadowing` 카테고리의 로어북 항목은 키워드 매칭 없이 항상 시스템 프롬프트에 포함됩니다.

---
### 7-4. 로어북 (Lorebook)

시나리오 세계관 정보를 항목별로 저장하고, 채팅 대화에서 키워드가 등장하면 해당 항목을 시스템 프롬프트에 자동 주입합니다.

**구현 위치**: `chat_engine_v2.py` → `extract_lorebook_entries()`, `get_lorebook_context()`

**로어북 항목 구조**:
```json
{
  "category": "character | setting | rule | item | foreshadowing",
  "name": "항목명",
  "keywords": ["트리거 키워드들"],
  "description": "상세 설명"
}
```

**키워드 매칭 로직**:
```python
# get_lorebook_context() 핵심 로직
# foreshadowing 카테고리: 항상 포함
# 나머지 카테고리: 유저/AI 최근 메시지에 keywords가 포함된 항목만 주입
if entry["category"] == "foreshadowing":
    always_include.append(entry)
elif any(kw in recent_text for kw in entry["keywords"]):
    context_entries.append(entry)
```

유저가 직접 항목을 추가·수정·삭제할 수 있습니다 (`PATCH /topics/{id}/lorebook/{index}`).

---
### 7-5. 인물 관계도

등장인물 간의 관계를 노드(인물)와 엣지(관계)로 표현하는 그래프입니다.

**구현 위치**: `chat_engine_v2.py` → `generate_relationship_graph()`

**갱신 시점**:
- 빌더 파이프라인 Step 5에서 초기 생성
- 기승전결 단계 전환 시 자동 갱신 (최근 10턴 원문 반영)
- 유저가 수동 새로고침 버튼 클릭 시

**채팅 프롬프트 반영**: 관계도가 매 턴 `build_chat_system_prompt()`에 주입되어 AI가 현재 인물 관계를 인식하고 응답합니다.

---
### 7-6. AI 추천 답변 / 힌트 카드

#### AI 추천 답변

유저가 어떻게 응답할지 막막할 때 AI가 3가지 답변 방향을 제안합니다.

- **엔드포인트**: `GET /topics/{id}/suggest-replies`
- **구현**: 현재 서사 단계와 최근 대화를 기반으로 Gemini가 3가지 선택지 생성
- **캐싱**: 같은 메시지에 대한 추천 답변은 캐싱 후 🔄 새로고침 버튼으로 재생성 가능

#### 힌트 카드

다음 서사 방향을 암시하는 짧은 힌트를 카드 형태로 제공합니다.

- **구현 위치**: `chat_engine_v2.py` → `generate_hint_card()`
- **모델**: Gemini 3.1 Flash-Lite (`gemini-3.1-flash-lite-preview`, max_tokens=50, 경량)
- **전달**: SSE `done` 이벤트의 `hint_card` 필드로 전송

---
### 7-7. 미디어 생성 (이미지 / BGM / 시네마틱)

#### 이미지 생성

| 기능 | 엔드포인트 | 모델 | 설명 |
|---|---|---|---|
| 캐릭터 이미지 | (빌더 내부) | Gemini 3.1 Flash Image | 캐릭터 프로필 이미지 |
| 배경 이미지 | `/topics/{id}/generate-background` | Gemini 3.1 Flash Image | 단계별 배경 |
| 단계 전환 캐릭터 | `/topics/{id}/generate-stage-character` | Gemini 3.1 Flash Image | 단계 전환 시 캐릭터 컷 |
| 엔딩 이미지 | `/topics/{id}/generate-ending-image` | Gemini 3.1 Flash Image | 엔딩 장면 |
| 채팅 표지 | (빌더 내부) | Gemini 3.1 Flash Image | ChatList 표지 이미지 |

모델 ID: `gemini-3.1-flash-image-preview` (Vertex AI)  
이미지는 Firebase Storage에 업로드하여 URL로 저장합니다.

#### BGM 생성

- **엔드포인트**: `POST /chat/generate-bgm`
- **모델**: Lyria 3 Pro (`lyria-3-pro-preview`, Vertex AI)
- **특징**: 장르·분위기·단계에 맞는 배경음악 생성. 직접 재생 가능.

#### 시네마틱 영상

- **엔드포인트**: `POST /topics/{id}/cinematic`
- **모델**: Veo 3.1 Fast (`veo-3.1-fast-generate-001`, Vertex AI)
- **특징**: 시나리오 **"기(도입부)" 단계**의 내용을 기반으로 짧은 영상 클립을 생성합니다. 채팅을 시작하기 전, 유저에게 어떤 배경·캐릭터·분위기의 이야기가 펼쳐질지 시각적으로 미리 보여주는 역할을 합니다.
- **프롬프트 구성**: 기 단계 시나리오 텍스트 + 세계관/배경 + 캐릭터 외형 정보 + 콘텐츠 유형별 스타일 힌트(실사 or 웹툰) + 톤·분위기에 따른 카메라 연출 힌트
- **보관함**: 생성된 영상은 `cinematic_urls` 컬럼에 누적 저장되며, 보관함에서 선택·삭제 가능

---
### 7-8. 인증·토큰·출석체크 시스템

#### Google 로그인 플로우

```
프론트엔드: Firebase SDK로 Google 소셜 로그인
    ↓ Firebase ID 토큰 발급
POST /auth/google {id_token}
    ↓ Firebase Admin SDK로 토큰 검증
백엔드: SQLite users 테이블에 google_id로 유저 생성 또는 조회
    ↓ 첫 로그인이면 is_new: true 반환 → 닉네임 입력 화면 표시
내부 JWT 발급 (PyJWT) → 이후 모든 API 호출에 Bearer 토큰으로 사용
```

#### DT(Dive Token) 시스템

| 항목 | 내용 |
|---|---|
| 신규 가입 기본 지급 | 10 DT |
| 일일 출석체크 기본 | +5,000 DT |
| 소비 | LLM 호출 기능 사용 시 모델별 차등 차감 |

#### 모델별 DT 소모량 (1회 응답당)

| 모델 | 소모량 |
|---|---|
| Gemini 3.1 Flash-Lite | 0 DT (무료) |
| Gemini 3 Flash | 25 DT |
| Gemini 2.5 Pro | 50 DT |
| Gemini 3.1 Pro | 90 DT |
| GPT-5.4 | 110 DT |
| 기능 갱신 (관계도·속마음 등) | 10 DT |

#### 30일 연속 출석체크 (Streak) — 누적 보너스

| 연속 일수 | 지급량 | 배율 |
|---|---|---|
| 1~2, 4~6, 8~13, 15~20, 22~29일 | 5,000 DT | ×1 |
| **3일** | 7,500 DT | ×1.5 |
| **7일** | 15,000 DT | ×3 |
| **14일** | 30,000 DT | ×6 |
| **21일** | 40,000 DT | ×8 |
| **30일** | 55,000 DT | ×11 |

- **엔드포인트**: `GET /auth/attendance` (현황 조회), `POST /auth/checkin` (체크인)
- **화면**: `AttendanceScreen.tsx` — 30일 캘린더 UI, 연속 일수 표시
- **구현 위치**: `main.py`, `models.py` (`consecutive_days`, `last_checkin_at` 컬럼)

In [ ]:
# auth.py 실제 구현 (핵심 로직)
import jwt
from datetime import datetime, timedelta

SECRET_KEY = "your-secret-key"  # 실제 환경에서는 .env에서 로드
ALGORITHM = "HS256"

def create_access_token_demo(user_id: int) -> str:
    """내부 JWT 발급 (auth.py 실제 구현)"""
    payload = {
        "sub": str(user_id),
        "exp": datetime.utcnow() + timedelta(days=30)
    }
    return jwt.encode(payload, SECRET_KEY, algorithm=ALGORITHM)

sample_token = create_access_token_demo(user_id=1)
print("JWT 구조 예시:")
decoded = jwt.decode(sample_token, SECRET_KEY, algorithms=[ALGORITHM])
print(f"  sub (user_id): {decoded['sub']}")
print(f"  exp (만료): {datetime.fromtimestamp(decoded['exp'])}")

---
### 7-9. 채팅방 복제

특정 AI 메시지 시점에서 대화를 복제하여 새로운 채팅방을 만드는 기능입니다.  
"이 시점에서 다른 선택을 했다면?"이라는 흥미를 충족하며, 이야기가 종료된 후 특정 시점으로 돌아가고 싶을 때도 활용할 수 있습니다.

- **엔드포인트**: `POST /topics/{id}/duplicate?until_message_id={mid}`
- **동작**: 지정된 메시지 ID까지의 모든 대화 내용, game_state, compass를 복사하여 새 토픽 생성
- **호감도 재계산**: 복제 시점까지의 대화를 재구성하여 그 시점의 `game_state.affinity`를 산출하고, 이를 기반으로 `affection`을 새로 계산합니다. 원본 채팅방의 호감도를 그대로 복사하지 않습니다.
- **프론트엔드**: `ChatInterface.tsx`에서 AI 메시지 왼쪽의 GitFork 아이콘 버튼 호버 시 **"채팅방 복제"** 툴팁 표시. 클릭 시 확인 오버레이("채팅방 복제 / 지금 복제하기") 등장. 복제 생성 후 system 메시지 자동 추가

### 7-10. 속마음 보기

AI 캐릭터가 현재 대화 상황에서 실제로 어떤 감정·생각을 갖고 있는지 보여주는 기능입니다.

- **엔드포인트**: `POST /topics/{id}/inner-thought`
- **다중 캐릭터 지원**: `character_name` 파라미터로 조연 캐릭터의 속마음도 확인 가능
- **자동 갱신**: 기→승→전→결 단계 전환 시마다 자동 업데이트

### 7-11. 대화 요약

채팅이 길어질 때 LLM이 과거 대화를 기억할 수 있도록 자동 요약을 생성합니다.

- **구현 위치**: `memory.py` → `check_and_auto_summarize()`
- **주기**: **기→승→전→결 단계 전환 시마다** 자동 생성 (DB `summaries` 테이블에 저장)
- **채팅 반영**: `build_chat_system_prompt()`의 `[이전 대화 요약]` 섹션에 자동 포함

---
### 7-11. 다시 하기

현재 시나리오를 처음부터 새로 플레이하는 기능입니다.  
기존 채팅방은 삭제되지 않고 그대로 유지되며, 별도의 새 채팅방이 생성됩니다.

- **엔드포인트**: `POST /topics/{id}/replay`
- **UI 위치**: 채팅 화면 헤더의 **"다시 하기"** 버튼 (RotateCcw 아이콘)
- **동작**:
  1. "다시 하기" 버튼 클릭 → 안내 모달 등장 ("현재 채팅방은 그대로 유지되고, 새로운 채팅방이 생성됩니다.")
  2. 유저가 **스토리 길이(단편/중편/장편)** 선택
  3. 확인 시 새 채팅방 생성 (시나리오·캐릭터·로어북·나침반은 초기 상태로 복사, 대화 이력 없음)
  4. 자동으로 새 채팅방으로 이동

- **복제와의 차이**:

| 항목 | 채팅방 복제 (7-9) | 다시 하기 (7-11) |
|---|---|---|
| 시작 시점 | 특정 메시지 시점 | 처음부터 |
| 대화 이력 | 복제 시점까지 복사 | 없음 |
| 목적 | 분기 탐색 | 새 플레이스루 |
| 스토리 길이 선택 | 불가 | 가능 |


---
### 7-12. 답변 재생성

AI 응답이 마음에 들지 않을 때 같은 유저 메시지에 대해 새로운 답변을 다시 생성합니다.

- **프론트엔드**: AI 메시지 왼쪽 RotateCcw 아이콘 버튼 호버 시 **"답변 재생성"** 툴팁 표시. 클릭 시 재생성 오버레이 등장 — 방향 가이드 텍스트(선택 입력) 후 "재생성하기"
- **버전 관리**: `is_regeneration: true` 플래그로 요청 → 기존 AI 응답 `is_active=False` 처리 후 `version+1`로 새 응답 저장. 재생성 횟수만큼 버전이 쌓이며 화면에서 **"1/2 ◀ ▶"** 형태로 버전 전환 가능
- **엔드포인트**: `POST /chat/stream` (`is_regeneration: true`, `guidance` 포함)
- **방향 가이드**: 재생성 시 원하는 응답 방향을 텍스트로 입력하면 시스템 프롬프트에 `[사용자 지시사항]`으로 주입됨

---
### 7-13. 대화 삭제

특정 메시지 시점 이후의 모든 대화를 삭제합니다.

- **프론트엔드**: 메시지 삭제 버튼 클릭 시 확인 모달 ("이 시점 이후의 모든 대화 내역이 삭제됩니다. AI도 해당 내용을 기억하지 못하게 됩니다.") → 확인 시 삭제
- **엔드포인트**: `DELETE /messages/{message_id}`
- **동작**: 선택한 메시지의 `created_at` 기준으로 같은 토픽의 **해당 시점 이후 모든 메시지**를 DB에서 완전 삭제 → AI가 삭제된 내용을 기억하지 못하는 상태로 복원

---
### 7-14. AI 성향

AI 캐릭터의 이번 답변 성향을 유저가 직접 설정합니다.

- **프론트엔드**: 채팅 입력창 하단 **"🎭 AI 성향"** 드롭다운

| 옵션 | 설명 | 지속성 |
|---|---|---|
| 없음 | 기본 페르소나 그대로 | - |
| 😊 긍정적 | 호의적·밝은 반응 | 해제 전까지 지속 |
| 😐 중립적 | 감정 억제, 담담한 반응 | 해제 전까지 지속 |
| 😤 부정적 | 냉담·거리두는 반응 | 해제 전까지 지속 |
| 🎲 Random | 주사위(1~6) 무작위 성향 | **이번 한 턴만** |

- **저장**: 없음/긍정/중립/부정 선택 시 `PATCH /topics/{id}` → `tone_preference` 컬럼에 저장 (채팅방에 영구 적용)
- **프롬프트 반영**: `tone_preference` 값이 `build_chat_system_prompt()`에 파라미터로 전달되어 AI 응답 스타일에 반영됨

---
### 7-15. 자동 진행

유저 입력 없이 AI가 지정한 턴 수만큼 이야기를 자동으로 이어갑니다.

- **프론트엔드**: 채팅 입력창 하단 **"⚡ 자동 진행"** 드롭다운 → **1턴 / 2턴 / 3턴** 선택
- **동작 흐름**:
  1. 프론트엔드가 `auto_advance: true` 플래그와 함께 요청 전송
  2. 백엔드에서 직전 AI 대사를 참고해 **LLM이 유저 캐릭터의 다음 행동/대사를 1~2문장으로 별도 생성** → 이 텍스트가 실제 user_message로 교체됨
  3. 교체된 유저 행동을 SSE `user_action` 이벤트로 먼저 전송
  4. 시스템 프롬프트에 `[자동 진행 중] 최근 대화 흐름을 참고하여 서사를 자연스럽게 이어갈 것. 등장 인물들을 상황에 맞게 활용해 장면을 풍성하게 구성할 것.` 추가 후 AI 응답 생성
  5. 각 턴 완료 후 남은 턴 자동 실행 (`autoTurnsRef`로 잔여 횟수 관리)
- **생성 실패 시 폴백**: `"[유저 캐릭터명]은 그 말을 듣고 잠시 멈추었다."`
- **제한**: 이야기가 종료(`is_ended=True`)된 상태에서는 자동 진행 버튼 비활성화

---
### 7-16. 북마크

채팅방 목록에서 특정 채팅방을 즐겨찾기로 표시하는 기능입니다.

- **구현 위치**: `ChatList.tsx`
- **저장 방식**: 브라우저 **localStorage** (`dive_bookmarks` 키) — 서버 DB가 아닌 클라이언트 로컬 저장
- **동작**:
  - 채팅방 목록에서 북마크 아이콘 버튼 클릭 → `toggleBookmark()` 호출 → `loadBookmarks()` / `saveBookmarks()`로 localStorage 업데이트
  - 북마크된 채팅방은 목록 **최상단**으로 정렬되고, **"북마크"** 섹션 구분선 아래에 별도 표시
  - 북마크 아이콘이 채워진 형태(filled)로 시각적 구분

---
### 7-17. 채팅 모델 선택

채팅방마다 사용할 AI 모델을 개별적으로 설정하는 기능입니다.

- **구현 위치**: `ChatInterface.tsx`
- **UI**: 채팅 화면 상단의 **"AI 모델 선택"** 버튼 클릭 시 모달 팝업 등장

| 모델 | 소모량 |
|---|---|
| Gemini 3.1 Flash-Lite | 무료 (0 DT) |
| Gemini 3 Flash | 25 DT |
| Gemini 2.5 Pro | 50 DT |
| Gemini 3.1 Pro | 90 DT |
| GPT-5.4 | 110 DT |

- **저장 방식**: 선택한 모델을 `localStorage`의 `dive_chat_model_{topic_id}` 키에 저장 → 채팅방별 모델 독립 유지
- **초기값**: 빌더 Step 5(세션 옵션)에서 선택한 모델(`sessionOptions?.model`)이 기본값으로 설정

---
### 7-18. DT 소모량 확인

해당 채팅방에서 지금까지 소모한 DT 총량과 항목별 내역을 확인하는 기능입니다.

- **구현 위치**: `ChatInterface.tsx`
- **UI**: 채팅 화면 상단 ⚡ **"DT 소모량"** 버튼 클릭 시 팝업 표시 (`showTokenPopup` 상태)
- **데이터 출처**: 채팅 메시지의 `allMessagesForUsage` 배열 — 각 메시지에 `spent_dt`(소모량), `model_name`(모델/기능명) 필드 포함
- **표시 내역**:

| 구분 | 조건 | 예시 |
|---|---|---|
| **총 소모량** | 전체 `spent_dt` 합산 | "총 1,250 DT" |
| **모델별 소모량** | `model_name`이 `FEATURE_`로 시작하지 않는 항목 | Gemini Flash: 500 DT, GPT-5.4: 220 DT |
| **기능별 소모량** | `model_name`이 `FEATURE_`로 시작하는 항목 | 속마음: 30 DT, 요약: 20 DT, 인물관계도: 40 DT, 추천답변: 10 DT |

---
### 7-19. 앨범(Album)

채팅 진행 중 생성된 모든 이미지를 한 곳에서 모아보는 탭입니다.

- **구현 위치**: `ChatInterface.tsx` — 사이드바 **Album** 탭 (ImageIcon 아이콘)
- **보관 이미지 종류**:

| 항목 | 생성 시점 | 비고 |
|---|---|---|
| **Cover** | 빌더 파이프라인 | 채팅방 목록에 표시되는 표지 이미지 |
| **AI 캐릭터 이미지** | 빌더 파이프라인 | AI 캐릭터 프로필 이미지 |
| **유저 캐릭터 이미지** | 빌더 파이프라인 | 유저 캐릭터 프로필 이미지 |
| **승/전/결 씬 캐릭터 컷** | 단계 전환 시 자동 생성 | 여러 장 누적 보관, ‹›로 탐색 가능 |
| **Ending** | 엔딩 발생 시 자동 생성 | 재생성·삭제 가능 |
| **❤️ 호감도 특전 CG** | 호감도 +100 달성 시 자동 생성 | 재생성·삭제 가능 |


#### 이미지 재생성

| 기능 | 버튼 위치 | 설명 |
|---|---|---|
| **이미지 세트 재생성** | 앨범 탭 상단 RotateCcw 버튼 | 표지·AI 캐릭터·유저 캐릭터 이미지를 한 번에 새로 생성 |
| **표지 단독 재생성** | Cover 섹션 "재생성" 버튼 | 표지 이미지만 단독으로 재생성 (100DT) |
| **캐릭터 이미지 재생성** | AI / 유저 캐릭터 섹션 "재생성" 버튼 | 해당 캐릭터 이미지만 재생성 (100DT) |

재생성된 이미지는 기존 이미지와 함께 **히스토리**로 누적 보관되며, ‹ › 버튼으로 슬라이드하여 원하는 이미지를 선택 후 "이 이미지로 설정" 버튼으로 대표 이미지 교체 가능합니다.  
불필요한 이미지는 "삭제" 버튼으로 히스토리에서 제거할 수 있습니다.

- **라이트박스**: 이미지 클릭 시 전체 화면 확대 (라이트박스 오버레이). 클릭 영역 외 클릭 또는 ✕ 버튼으로 닫기

---
### 7-20. 호감도 (Affinity)

AI 캐릭터가 유저 캐릭터에 대해 갖는 감정 수치를 나타냅니다.

**구현 위치**: `chat_engine_v2.py` (GameStateV2), `ChatInterface.tsx` (UI 표시)

#### 수치 시스템

- **범위**: -100 ~ +100 (정수)
- **매 턴 변화**: LLM이 반환한 `affinity_delta` (+/- 정수)를 누적 합산
- **단계별 delta cap**: 기 ±3 / 승 ±5 / 전 ±8 (급격한 변화 방지)

#### UI 표시 위치

**① 사이드바 Status 탭 — 등장인물 카드**

| 상태 | 표시 내용 |
|---|---|
| 카드 **접힌** 상태 | 숫자값 (`+값` / `-값`) |
| 카드 **펼친** 상태 | 게이지 바 + 수치 + 텍스트 레이블 |

**호감도 레이블 색상**:

| 범위 | 텍스트 레이블 | 색상 |
|---|---|---|
| +60 ~ +100 | 매우 우호적 | 보라(violet) |
| +20 ~ +59 | 우호적 | 파랑(blue) |
| -19 ~ +19 | 중립 | 슬레이트(slate) |
| -60 ~ -20 | 경계심 | 주황(orange) |
| -100 ~ -61 | 적대적 | 빨강(red) |

게이지 바 너비: `affection = (affinity + 100) / 2` 변환(0~100%)으로 계산

**② 헤더 오른쪽** — 보유 DT 옆에 표시 없음(수치 표시는 사이드바에만)

#### 호감도 +100 특전

- 채팅 중 호감도가 +100에 처음 도달하면 백엔드에서 **특전 씬 텍스트 자동 생성**
- SSE done 이벤트의 `affinity_max_scene` 필드로 전달 → 프론트엔드에서 전면 오버레이 팝업 표시
- 동시에 `/topics/{id}/generate-affinity-image` 호출로 **특전 CG 이미지 자동 생성**
- 생성된 특전 CG는 갤러리 탭 "❤️ 호감도 특전" 섹션에 보관

---
### 7-21. 유저 노트

유저가 AI에게 지속적으로 전달하고 싶은 지시사항을 채팅방별로 저장·관리하는 기능입니다.

- **구현 위치**: `ChatInterface.tsx`, `chat_engine_v2.py`
- **UI 위치**: 사이드바 Status 탭 하단 **"유저 노트"** 섹션 (보라색 구분선)

#### 노트 프리셋 시스템

여러 개의 노트를 저장해두고 필요에 따라 전환하여 적용할 수 있습니다.

- **노트 추가**: "+" 버튼 → 제목 + 내용 입력 폼 등장
- **노트 적용**: "적용" 버튼 클릭 → 해당 노트 내용이 현재 활성 유저 노트로 설정
- **적용 표시**: 현재 적용 중인 노트는 보라색 배경 하이라이트 + "✓ 현재 적용 중" 문구
- **노트 수정/삭제**: 각 노트 카드에 편집(✏️)·삭제(🗑️) 버튼

#### 시스템 프롬프트 주입

```python
# chat_engine_v2.py — build_chat_system_prompt()
user_notes_section = f'[유저 노트(항상 기억)] {user_notes}\n' if user_notes else ''
# → 매 턴 시스템 프롬프트에 포함되어 AI가 지시사항을 항상 인식
```

#### 저장 방식

- **엔드포인트**: `PATCH /topics/{id}` → `user_notes` 컬럼 (현재 적용 중인 내용) + `user_note_presets` 컬럼 (프리셋 목록 JSON)

---
### 7-22. 기승전결 진행 현황

현재 이야기가 기/승/전/결 중 어느 단계인지를 두 곳에서 시각적으로 표시합니다.

**구현 위치**: `ChatInterface.tsx` — `currentStage` 상태, SSE `meta.stage`로 업데이트

#### ① 채팅 화면 헤더

채팅 제목 오른쪽에 기-승-전-결 4개 원형 뱃지가 연결바와 함께 표시됩니다.

| 상태 | 스타일 |
|---|---|
| **현재 단계** | 보라-인디고 그라디언트 배경, 흰색 텍스트, 강조 그림자 |
| **지나간 단계** | 희미한 보라 배경, 보라색 텍스트 |
| **미래 단계** | 투명 배경, 회색(비활성) 텍스트 |

#### ② 사이드바 Status 탭 최상단

기/승/전/결 4개 블록이 가로로 나란히 표시됩니다.

| 상태 | 스타일 |
|---|---|
| **현재 단계** | 보라-인디고 그라디언트 + 아래 보라색 인디케이터 바 |
| **지나간 단계** | 어두운 보라 배경 + 흐린 보라 인디케이터 |
| **미래 단계** | 회색 배경, 비활성 |

#### 단계 전환 흐름

1. LLM이 `trigger_branch: true` 반환
2. 백엔드 Stage Gate 통과 (`stage_turn_count >= MIN_STAGE_TURNS`)
3. `generate_next_stage()` 호출 → 단계 전환 오프닝 텍스트 생성
4. SSE done 이벤트의 `meta.stage`로 프론트엔드에 새 단계 전달
5. `currentStage` 상태 즉시 갱신 → UI 자동 업데이트

---
### 7-23. 전체 턴 수 / 단계별 턴 수

**구현 위치**: `chat_engine_v2.py` — `GameStateV2` 내 두 가지 턴 카운터

| 필드 | 설명 | 초기화 시점 |
|---|---|---|
| `total_turn_count` | 전체 대화 누적 턴 수 | 채팅방 생성 시 0 |
| `stage_turn_count` | 현재 단계 내 턴 수 | 단계 전환마다 0으로 리셋 |

#### 활용 방식

```python
# chat_engine_v2.py

# 1. 매 턴마다 카운터 증가
game_state.total_turn_count += 1
game_state.stage_turn_count += 1

# 2. LLM 시스템 프롬프트에 현황 주입 (AI가 서사 흐름 인식)
f'단계: {current_stage} ({stage_turn_count}턴) | 전체: {total_turn_count}턴 | AI 호감도: {affinity:+d}'

# 3. 내부 대화 요약 자동 생성 (SUMMARY_INTERVAL = 10)
if total_turn_count > 0 and total_turn_count % 10 == 0:
    game_state.conversation_summary = summarize_conversation(...)

# 4. Stage Gate — 최소 턴 미충족 시 단계 전환 차단
stage_turn_count < MIN_STAGE_TURNS[current_stage]  # → trigger_branch 강제 False
```

#### 단계별 최소 턴 수 (MIN_STAGE_TURNS)

```python
# chat_engine_v2.py
MIN_STAGE_TURNS = {'기': 10, '승': 12, '전': 7, '결': 3}
```

| 단계 | 최소 턴 수 |
|---|---|
| 기 | 10턴 |
| 승 | 12턴 |
| 전 | 7턴 |
| 결 | 3턴 |
| **합계 (최소 완주)** | **32턴** |

#### 실제 플레이 턴 수 통계 (로컬 프로토타입 기준)

SQLite DB (`sql_app.db`) 내 v2 채팅 세션 데이터를 기반으로 집계한 결과입니다.

| 구분 | 세션 수 | 평균 턴 수 | 범위 |
|---|---|---|---|
| **완결된 세션** (엔딩 도달) | 3개 | **45.7턴** | 45 ~ 47턴 |
| **진행 중 세션** | 6개 | 8.0턴 | 1 ~ 20턴 |
| **전체 평균** | 9개 | 20.6턴 | 1 ~ 47턴 |

- **완결 세션 상세**: 3개 모두 결 단계에서 종료 / 호감도 +100(해피 엔딩) 2건, +17(평범한 엔딩) 1건
- **완결 세션 턴 수**: 현재 MIN_STAGE_TURNS 기준 최소 완주 32턴 대비 약 **1.4배** 플레이 후 엔딩 도달
- **프론트엔드 표시**: 직접 숫자로 표시되지 않음. 백엔드 게임 상태 전용 내부 카운터
- **채팅방 복제 시**: 복제 시점까지의 `total_turn_count` / `stage_turn_count`를 메시지 기록에서 재계산하여 cloned_game_state에 복원

---
### 7-24. 단계 전환 (분기)

LLM이 서사 흐름을 스스로 판단하여 기→승→전→결 순서로 단계를 넘기는 자동 전환 메커니즘입니다.

**구현 위치**: `chat_engine_v2.py` → `generate_next_stage()`

#### 전환 조건 — 두 가지를 동시에 충족해야 함

| 조건 | 설명 |
|---|---|
| **LLM 신호** | LLM 응답 JSON에 `trigger_branch: true` 포함 |
| **Stage Gate** | `stage_turn_count >= MIN_STAGE_TURNS[current_stage]` |

Stage Gate는 너무 이른 단계 전환을 방지합니다. 최소 턴 미충족 시 `trigger_branch`가 `true`여도 강제로 `false` 처리합니다.

```python
# chat_engine_v2.py — Stage Gate 로직
if game_state.stage_turn_count < MIN_STAGE_TURNS.get(game_state.current_stage, 0):
    trigger_branch = False   # 최소 턴 미충족: 전환 차단
    trigger_ending = False
```

#### generate_next_stage() 동작

| 항목 | 내용 |
|---|---|
| **모델** | MODEL_PRO (`gemini-3.1-pro-preview`) |
| **파라미터** | temperature=0.75, max_tokens=500 |
| **출력 길이** | 200~400자 단계 전환 오프닝 텍스트 |

**단계별 특수 규칙**:

| 이동 대상 단계 | 프롬프트 규칙 |
|---|---|
| **전(클라이맥스 직전)** | 오프닝에서 클라이맥스를 바로 드러내지 않음. 긴장감과 복선만 제시 |
| **결(엔딩 직전)** | 선택 분기 상황만 제시하고 결말은 유저에게 열어둠 |

#### 전환 후 처리 흐름

```
generate_next_stage() 완료
  ↓
current_stage 갱신 (기→승, 승→전, 전→결)
stage_turn_count = 0  (리셋)
  ↓
generate_relationship_graph()  [관계도 자동 갱신]
  ↓
단계 전환 씬 캐릭터 이미지 생성 요청 (비동기)
  ↓
SSE done 이벤트에 stage_opening 필드로 오프닝 텍스트 전송
  ↓
프론트엔드: is_stage_opening: true 메시지로 채팅창에 표시
           currentStage 상태 즉시 갱신 → 헤더·사이드바 UI 업데이트
```

#### DB 저장 형식

```python
# 단계 전환 메시지는 아래 구조로 messages 테이블에 저장됨
{
  "reply": "<오프닝 텍스트>",
  "is_stage_opening": True,
  "stage": "<새 단계>"   # "승" / "전" / "결"
}

---
### 7-25. 엔딩

결 단계에서 LLM이 `trigger_ending: true`를 반환하면 엔딩을 자동 생성합니다.  
엔딩의 색채와 결말 방향은 지금까지의 대화 흐름과 시나리오 맥락에서 자연스럽게 도출됩니다.

**구현 위치**: `chat_engine_v2.py` → `generate_ending_scene()`

#### 엔딩 발동 조건

```python
# trigger_ending은 trigger_branch도 true여야 함 (결 단계 전환과 동시 또는 결 단계 내에서)
# Stage Gate를 통과한 이후에만 유효 (result 단계 최소 3턴 이상)
if trigger_ending and game_state.current_stage == "결":
    ending = await generate_ending_scene(game_state, ...)
```

#### generate_ending_scene() 동작

| 항목 | 내용 |
|---|---|
| **모델** | MODEL_PRO (`gemini-3.1-pro-preview`) |
| **파라미터** | temperature=0.7, max_tokens=600 |
| **출력 길이** | 300~500자 엔딩 씬 텍스트 |
| **엔딩 방향** | 해피·배드 유형 구분 없이 대화 흐름과 시나리오 맥락에서 자연스럽게 결말 도출 |
| **반환값** | `{'type': '', 'scene': scene, 'affinity': affinity}` |

#### 엔딩 후 처리 흐름

```
generate_ending_scene() 완료
  ↓
game_state.is_ended = True  → 이후 추가 채팅 차단
  ↓
SSE done 이벤트에 ending 필드로 엔딩 씬 전송
  ↓
프론트엔드: is_ending: true 메시지로 채팅창에 엔딩 카드 표시
           ChatList에 "완결" 뱃지 표시
  ↓
엔딩 이미지 자동 생성 (/topics/{id}/generate-ending-image)
  → 생성된 이미지는 갤러리 탭 "Ending" 섹션에 보관
```

#### DB 저장 형식

```python
# 엔딩 메시지는 아래 구조로 messages 테이블에 저장됨
{
  "reply": "<엔딩 씬 텍스트>",
  "is_ending": True,
  "ending_type": ""
}
```

#### 엔딩 카드 UI (ChatInterface.tsx)

- 채팅창에 전면 엔딩 카드로 표시 (씬 텍스트 + 생성된 엔딩 이미지)
- 엔딩 발생 후 입력창 비활성화 — 단, **채팅방 복제**로 엔딩 이전 시점으로 되돌아가 다른 선택을 시도할 수 있음

---
### 7-26. 게시 기능

완성된 시나리오를 시나리오 갤러리에 공개 게시하는 기능입니다.

- **엔드포인트**: `POST /topics/{id}/publish` (토글 — 게시/게시 취소 반복)
- **게시 조건**: 나침반(compass)이 생성된 시나리오만 게시 가능
- **UI 위치**: 채팅 목록(`ChatList.tsx`) 각 채팅방 행의 **BookOpen 아이콘** 버튼
  - 나침반이 없는 채팅방에는 버튼 미표시
  - 갤러리에서 가져온(`imported_from_id` 존재) 채팅방에도 버튼 미표시
- **게시 확인 모달**: "갤러리 게시 / 이 시나리오를 갤러리에 공개할까요?" 확인 후 실행
- **게시 상태 표시**: 채팅 목록 뱃지에 **"게시중"** (초록색) 표시, 게시 취소 시 제거
- **갤러리 공개 데이터**: 제목, 표지 이미지, 캐릭터 이미지, 장르, 인트로, 나침반 정보


---
### 7-27. 시나리오 갤러리

다른 유저가 배포한 시나리오를 탐색하고 내 채팅방으로 가져오는 기능입니다.

- **구현 위치**: `ScenarioGallery.tsx`
- **엔드포인트**: `GET /scenarios/published`

#### 탐색 및 필터

| 필터 | 설명 |
|---|---|
| **콘텐츠 유형 칩** | 만화·소설·시리즈·영화·고전 필터링 |
| **국가 칩** (고전 전용) | 한국·중국·일본 필터링 |
| **장르 칩** | 선택한 유형/국가에 맞는 장르 필터링 |
| **검색** | 제목·작가명·인트로·장르 키워드 검색 |

#### 시나리오 가져오기

| 버튼 | 동작 |
|---|---|
| **이 시나리오 담기** | 스토리 길이 선택 모달 → 채팅 목록에 추가 (채팅 이동 X) |
| **바로 시작하기** | 스토리 길이 선택 모달 → 채팅 목록에 추가 + 즉시 채팅 이동 |

- **스토리 길이 선택 모달**: 단편 / 중편 / 장편 중 선택 후 가져오기 확정
- **엔드포인트**: `POST /scenarios/{id}/import` (`story_length` 파라미터 포함)
- **이미지 처리**: 표지·AI 캐릭터·유저 캐릭터 이미지를 Firebase에 독립 복제 (원작자 시나리오 삭제 시에도 이미지 유지)
- **원작자 표시**: 채팅 목록 및 채팅 정보 모달에 **"by 작가명"** 뱃지 표시


---
### 7-28. 팔로잉 / 팔로워

갤러리에서 마음에 드는 작가를 팔로우하는 소셜 기능입니다.

- **엔드포인트**:
  - `POST /users/{id}/follow` — 팔로우
  - `DELETE /users/{id}/follow` — 팔로우 취소
- **UI 위치**: 시나리오 갤러리 카드 내 작가명 옆 **팔로우 / 팔로잉** 버튼
  - 본인 시나리오에는 버튼 미표시
  - 팔로잉 상태: 보라색 「팔로잉」 버튼 (클릭 시 취소)
  - 미팔로우 상태: 회색 「팔로우」 버튼
- **저장**: `follows` 테이블 (follower_id, following_id)
- **팔로워/팔로잉 수**: 작가 프로필 페이지에서 확인 가능


---
### 7-29. 작가 프로필

배포 시나리오 작가의 프로필 페이지입니다.

- **구현 위치**: `AuthorProfile.tsx`
- **진입 경로**: 시나리오 갤러리 카드의 **작가명 클릭**
- **표시 정보**:
  - 작가 닉네임
  - 팔로워 수 / 팔로잉 수
  - 팔로우 / 팔로잉 버튼 (본인 프로필 제외)
  - 해당 작가가 배포한 시나리오 목록
- **엔드포인트**:
  - `GET /users/{id}/profile` — 프로필 + 팔로워·팔로잉 수
  - `GET /users/{id}/scenarios` — 배포 시나리오 목록


---
### 7-30. 스토리 길이 선택 (단편 / 중편 / 장편)

시나리오를 몇 턴에 걸쳐 진행할지 선택하는 기능입니다.

#### 선택 시점

| 시점 | 위치 |
|---|---|
| **새 시나리오 생성** | Screen4 — "플레이 시작" 버튼 위 선택기 |
| **다시 하기** | 다시 하기 모달 (7-11) |
| **갤러리 가져오기** | 가져오기 모달 (7-27) |

#### 길이별 스펙

| 선택지 | 내부값 | 평균 턴 수 | 기 | 승 | 전 | 결 |
|---|---|---|---|---|---|---|
| **단편** | `short` | ~20턴 | 5 | 6 | 4 | 2 |
| **중편** | `normal` | ~40턴 | 10 | 12 | 7 | 3 |
| **장편** | `long` | ~80턴 | 20 | 25 | 15 | 5 |

```python
# chat_engine_v2.py
STAGE_TURNS_BY_LENGTH = {
    'short':  {'기': 5,  '승': 6,  '전': 4,  '결': 2},
    'normal': {'기': 10, '승': 12, '전': 7,  '결': 3},
    'long':   {'기': 20, '승': 25, '전': 15, '결': 5},
}
```

#### 저장 및 표시

- **저장**: `game_state.story_length` 필드 (JSON)
- **채팅 목록 뱃지**: 단편(주황) / 중편(회색) / 장편(파란색) 뱃지로 시각 구분
- **기존 채팅방**: story_length 미설정 시 중편으로 표시 (기본값)


---
## 8. 프론트엔드 화면 구성

**구성**: React 18 + TypeScript + Vite + Tailwind CSS  
**상태 관리**: `App.tsx`에서 `FlowData` 상태로 빌더 플로우 전체 관리

### 빌더 플로우 (Screen0 → Screen5)

| 화면 | 파일 | 설명 |
|---|---|---|
| **Screen0** | `Screen0.tsx` | 홈 랜딩 — "새로운 모험 시작하기" 버튼 |
| **Screen1** | `Screen1.tsx` | 콘텐츠 유형(소설/만화 등) + 장르 선택 |
| **Screen2** | `Screen2.tsx` | 소재 입력 (빈칸 = AI 자동 생성) + 캐릭터 설정 |
| **Screen3** | `Screen3.tsx` | SSE 파이프라인 실행 — 단계별 로딩 표시. 생성 중 탭 이탈 방지, 오류 시 재시도 버튼 |
| **Screen4** | `Screen4.tsx` | 생성 결과 확인 — 시나리오·캐릭터 카드(성별·나이 포함)·시작 배경 표시 |
| **Screen5** | `Screen5.tsx` | 세션 옵션 설정 — AI 모델 선택, 응답 길이, 기타 설정 |

### 채팅 화면

| 화면 | 파일 | 주요 UI 요소 |
|---|---|---|
| **ChatInterface** | `ChatInterface.tsx` | 채팅창, 사이드바(나침반/로어북/관계도/속마음/요약), 힌트 카드, 호감도 게이지, 단계 표시, 자동 진행(1~3턴), 추천 답변, 응답 길이 설정, 오류 토스트, 엔딩 카드, 배경·BGM·시네마틱 |
| **ChatList** | `ChatList.tsx` | 채팅방 목록 — 검색, 북마크, 표지 이미지, 완결 표시, 비로그인 안내 |

### 기타 화면

| 화면 | 파일 | 설명 |
|---|---|---|
| **ScenarioGallery** | `ScenarioGallery.tsx` | 시나리오 갤러리 — 콘텐츠 유형/장르/국가 필터, 검색, 팔로우, 가져오기 |
| **AuthorProfile** | `AuthorProfile.tsx` | 작가 프로필 — 배포 시나리오 목록, 팔로워/팔로잉 수 |
| **Settings** | `Settings.tsx` | 마이페이지 — Google 로그인, 닉네임 설정, 보유 DT 표시, 출석체크, 로그아웃 |
| **AttendanceScreen** | `AttendanceScreen.tsx` | 30일 연속 출석체크 전용 화면 — 캘린더 UI, 연속 일수(streak) |
| **LoginScreen** | `LoginScreen.tsx` | Google 소셜 로그인 화면 |
| **NicknameScreen** | `NicknameScreen.tsx` | 첫 로그인 시 닉네임 입력 화면 |

---
## 9. 결론

### 구현 완료 사항

| 영역 | 주요 성과 |
|---|---|
| **데이터** | AI Hub 두 데이터셋 89,851개 씬 + 12,717개 고전 단락을 ChromaDB RAG로 구축 |
| **AI 파이프라인** | 6단계 SSE 빌더 + GameStateV2 기반 기승전결 채팅 엔진 완성 |
| **서사 품질** | 나침반·로어북·관계도·요약이 협력하여 긴 대화에서도 서사 일관성 유지 |
| **미디어** | Gemini 3.1 Flash Image + Lyria 3 Pro BGM + Veo 3.1 Fast 시네마틱 영상 생성 통합 |
| **인증** | Firebase Google 로그인 + JWT + DT 토큰 + 30일 streak 출석 시스템 |
| **UX** | 추천 답변, 힌트 카드, 속마음 보기, 대화 분기 등 풍부한 보조 기능 |
| **소셜** | 시나리오 갤러리 배포·가져오기, 작가 팔로우, 작가 프로필 |
| **플레이 옵션** | 스토리 길이(단편/중편/장편) 선택, 다시 하기, 갤러리 가져오기 시 이미지 독립 복제 |
| **앨범** | 표지·캐릭터 이미지 개별 재생성 및 히스토리 관리 |